# 02. Analisis del Piloto

## Objetivo

Evaluar el desempeño del piloto de tarjetas de débito físicas y comparar el comportamiento de los clientes que recibieron una tarjeta física frente a quienes no la recibieron.

El análisis busca:

- identificar la adopción y activación de las tarjetas físicas;
- medir cambios en frecuencia y monto transaccional;
- comparar el comportamiento entre clientes con y sin tarjeta física;
- definir las bases analíticas que posteriormente permitirán construir un criterio de priorización para la siguiente ola de entrega.

## 0. Configuracion

In [7]:
# Imports
# Configura las librerías necesarias para leer los datasets Silver desde S3.

from io import BytesIO

import boto3
import pandas as pd

In [8]:
# Configuración AWS
# Define el perfil local y la ubicación de la capa Silver.

AWS_PROFILE = "ds-technical-test"
S3_BUCKET = "bg-ds-debit-card-pilot-bucket"
SILVER_PREFIX = "silver"

session = boto3.Session(profile_name=AWS_PROFILE)
s3 = session.client("s3")

In [9]:
# Lectura de datasets Silver
# Lee todos los archivos Parquet de un prefijo de S3 y los consolida en un DataFrame.

def read_s3_parquet_folder(folder: str) -> pd.DataFrame:
    prefix = f"{SILVER_PREFIX}/{folder}/"

    paginator = s3.get_paginator("list_objects_v2")
    parquet_files = []

    for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=prefix):
        parquet_files.extend(
            obj["Key"]
            for obj in page.get("Contents", [])
            if obj["Key"].endswith(".parquet")
        )

    dataframes = []

    for key in parquet_files:
        response = s3.get_object(Bucket=S3_BUCKET, Key=key)
        dataframes.append(
            pd.read_parquet(BytesIO(response["Body"].read()))
        )

    return pd.concat(dataframes, ignore_index=True)

In [10]:
# Carga de Silver
# Recupera las cinco fuentes procesadas y estandarizadas por el Glue ETL Job.

customers = read_s3_parquet_folder("customers")
cards = read_s3_parquet_folder("cards")
transactions = read_s3_parquet_folder("transactions")
marketing_interactions = read_s3_parquet_folder("marketing_interactions")
merchant_catalog = read_s3_parquet_folder("merchant_catalog")

In [11]:
# Validación de carga

datasets = {
    "customers": customers,
    "cards": cards,
    "transactions": transactions,
    "marketing_interactions": marketing_interactions,
    "merchant_catalog": merchant_catalog,
}

for name, df in datasets.items():
    print(f"{name}: {len(df):,} rows")

customers: 12,000 rows
cards: 16,585 rows
transactions: 193,015 rows
marketing_interactions: 36,000 rows
merchant_catalog: 42 rows


## 1. Definicion del Piloto

### 1.1. Identificacion de Tarjetas Fisicas y Virtuales

In [12]:
# Distribución de tipos de tarjeta

display(
    cards["tipo"]
    .value_counts(dropna=False)
    .to_frame("count")
)

,count
tipo,
virtual,11203
fisica,5382


In [ ]:
# # Validación de categóricas en Silver
# # Confirma que las homologaciones aplicadas dejaron dominios consistentes.

# categorical_checks = {
#     "customers": ["ciudad", "canal_adquisicion", "estado_cuenta"],
#     "cards": ["tipo"],
#     "transactions": ["tipo_transaccion", "es_devolucion"],
#     "marketing_interactions": ["campana", "canal", "respondio"],
#     "merchant_catalog": ["categoria"],
# }

# for name, columns in categorical_checks.items():
#     print(f"\n{name.upper()}")

#     for column in columns:
#         values = datasets[name][column].value_counts(dropna=False)
#         print(f"\n{column} ({len(values)} values)")
#         display(values)


CUSTOMERS

ciudad (9 values)


ciudad
quito         3842
guayaquil     3581
cuenca        1198
ambato         808
manta          749
machala        596
loja           465
portoviejo     386
riobamba       375
Name: count, dtype: int64


canal_adquisicion (5 values)


canal_adquisicion
organico              3926
publicidad_digital    3342
referido              2177
call_center           1715
NaN                    840
Name: count, dtype: int64


estado_cuenta (4 values)


...

canal
push        9084
email       9048
sms         9022
whatsapp    8846
Name: count, dtype: int64


respondio (3 values)


respondio
False    23130
True     10866
None      2004
Name: count, dtype: int64


MERCHANT_CATALOG

categoria (8 values)


categoria
streaming             12
supermercado           7
retail                 5
restaurante            5
entretenimiento        4
salud                  4
servicios_publicos     4
transporte             1
Name: count, dtype: int64

In [14]:
# Tenencia de tarjetas por cliente
# Clasifica a cada cliente según tenga tarjeta física, virtual o ambas.

card_ownership = (
    cards
    .assign(
        has_physical=cards["tipo"].eq("fisica"),
        has_virtual=cards["tipo"].eq("virtual"),
    )
    .groupby("cliente_id", as_index=False)
    .agg(
        has_physical=("has_physical", "max"),
        has_virtual=("has_virtual", "max"),
    )
)

card_ownership["card_group"] = "other"

card_ownership.loc[
    card_ownership["has_physical"] & ~card_ownership["has_virtual"],
    "card_group",
] = "physical_only"

card_ownership.loc[
    ~card_ownership["has_physical"] & card_ownership["has_virtual"],
    "card_group",
] = "virtual_only"

card_ownership.loc[
    card_ownership["has_physical"] & card_ownership["has_virtual"],
    "card_group",
] = "physical_and_virtual"

display(
    card_ownership["card_group"]
    .value_counts()
    .to_frame("customers")
)

,customers
card_group,
virtual_only,6309
physical_and_virtual,4894
physical_only,488


In [29]:
cards.head()

,cliente_id,tarjeta_id,tipo,estado_codigo,fecha_emision,fecha_activacion,is_orphan_customer
0,CHK-000117,TRJ-000166,fisica,1,2026-05-01,NaT,False
1,CHK-009066,TRJ-012529,virtual,1,2022-03-05,2022-03-11,False
2,CHK-003837,TRJ-005282,virtual,1,2026-05-18,2026-05-20,False
3,CHK-002434,TRJ-003366,virtual,2,2023-06-26,2023-07-03,False
4,CHK-003930,TRJ-005404,virtual,1,2021-12-23,2021-12-31,False


In [15]:
# Cobertura de clientes con tarjetas
# Verifica cuántos clientes del maestro aparecen en el inventario de tarjetas.

print(f"Customers: {customers['cliente_id'].nunique():,}")
print(f"Customers with cards: {card_ownership['cliente_id'].nunique():,}")
print(
    f"Customers without cards: "
    f"{customers['cliente_id'].nunique() - card_ownership['cliente_id'].nunique():,}"
)

Customers: 12,000
Customers with cards: 11,691
Customers without cards: 309


In [32]:
# Fechas de activación faltantes por tipo
# Verifica si los NaT de fecha_activacion se concentran únicamente en tarjetas físicas.

activation_missing_by_type = (
    cards
    .groupby("tipo")
    .agg(
        total_cards=("tarjeta_id", "size"),
        without_activation=("fecha_activacion", lambda x: x.isna().sum()),
        with_activation=("fecha_activacion", lambda x: x.notna().sum()),
    )
)

activation_missing_by_type["missing_pct"] = (
    activation_missing_by_type["without_activation"]
    / activation_missing_by_type["total_cards"]
    * 100
).round(2)

display(activation_missing_by_type)

,total_cards,without_activation,with_activation,missing_pct
tipo,,,,
fisica,5382,2809,2573,52.19
virtual,11203,0,11203,0.00


In [24]:
# Tarjetas sin fecha de activación
# Permite revisar su tipo, estado y fecha de emisión para interpretar el significado del NaT.

cards.loc[
    cards["fecha_activacion"].isna(),
    [
        "tarjeta_id",
        "cliente_id",
        "tipo",
        "estado_codigo",
        "fecha_emision",
        "fecha_activacion",
    ],
].head(10)

,tarjeta_id,cliente_id,tipo,estado_codigo,fecha_emision,fecha_activacion
0,TRJ-000166,CHK-000117,fisica,1,2026-05-01,NaT
17,TRJ-002668,CHK-001933,fisica,2,2026-05-01,NaT
20,TRJ-006970,CHK-005061,fisica,2,2026-05-01,NaT
27,TRJ-001241,CHK-000893,fisica,2,2026-05-01,NaT
29,TRJ-009280,CHK-006746,fisica,2,2026-05-01,NaT
32,TRJ-009278,CHK-006745,fisica,2,2026-05-01,NaT
34,TRJ-013349,CHK-009664,fisica,1,2026-05-01,NaT
35,TRJ-009881,CHK-007190,fisica,2,2026-05-01,NaT
37,TRJ-004703,CHK-003405,fisica,1,2026-05-01,NaT
44,TRJ-005682,CHK-004126,fisica,1,2026-06-24,NaT


In [31]:
# Distribución de estados por tipo de tarjeta

display(
    pd.crosstab(
        cards["tipo"],
        cards["estado_codigo"],
        margins=True,
    )
)

estado_codigo,1,2,3,All
tipo,,,,
fisica,4503,879,0,5382
virtual,9482,862,859,11203
All,13985,1741,859,16585


#### Observaciones

- De los 11,691 clientes presentes en Cards, 6,309 tienen únicamente tarjeta virtual, 488 únicamente tarjeta física y 4,894 poseen ambas.

- En total, 5,382 clientes tienen al menos una tarjeta física, mientras que 11,203 tienen al menos una tarjeta virtual.

- No se identificaron clientes clasificados en categorías distintas de física o virtual después de la homologación, lo que confirma que la estandarización de `tipo` quedó consistente.

- De los 12,000 clientes del maestro, 309 no aparecen en el inventario de tarjetas. Estos clientes deberán mantenerse diferenciados de quienes poseen únicamente tarjeta virtual, ya que representan un grupo sin tarjeta registrada.

- `fecha_activacion` presenta un comportamiento diferente según el tipo de tarjeta: las 11,203 tarjetas virtuales registran fecha de activación, mientras que únicamente 2,573 de las 5,382 tarjetas físicas (47.81%) la presentan. Por esta razón, la activación no se utilizará como criterio para definir la cohorte del piloto, sino como un evento posterior a la emisión.

- `estado_codigo` presenta valores numéricos sin un diccionario o definición de negocio disponible en los datos entregados. Por este motivo, no se utilizará para inferir estados como activa, inactiva o bloqueada, evitando introducir supuestos no sustentados.

- La emisión de tarjeta física se utilizará inicialmente para identificar la cohorte del piloto, mientras que la activación se analizará como un resultado posterior del proceso. La definición temporal del análisis se completará en la siguiente sección.

### 1.2. Cohortes y Temporalidad

In [16]:
# Campañas de marketing
# Revisa las campañas disponibles y su volumen de contactos.

display(
    marketing_interactions["campana"]
    .value_counts()
    .to_frame("contacts")
)

,contacts
campana,
reactivacion ahorros,7348
referidos q3,7243
piloto tarjeta fisica q2,7164
bienvenida nuevos usuarios,7145
cashback verano,7100


In [17]:
# Campaña del piloto de tarjeta física
# Identifica clientes contactados y rango temporal de la campaña.

pilot_campaign = marketing_interactions[
    marketing_interactions["campana"].eq("piloto tarjeta fisica q2")
].copy()

print(f"Contacts: {len(pilot_campaign):,}")
print(f"Unique customers: {pilot_campaign['cliente_id'].nunique():,}")
print(f"Start date: {pilot_campaign['fecha_contacto'].min()}")
print(f"End date: {pilot_campaign['fecha_contacto'].max()}")

display(
    pilot_campaign["respondio"]
    .value_counts(dropna=False)
    .to_frame("contacts")
)

Contacts: 7,164
Unique customers: 5,402
Start date: 2021-01-01 00:00:00
End date: 2026-06-30 00:00:00


,contacts
respondio,
False,4658
True,2094
None,412


In [18]:
# Temporalidad de tarjetas físicas
# Revisa cuándo fueron emitidas y activadas las tarjetas físicas.

physical_cards = cards[
    cards["tipo"].eq("fisica")
].copy()

print(f"Physical cards: {len(physical_cards):,}")
print(f"Customers with physical card: {physical_cards['cliente_id'].nunique():,}")
print(f"Issue date range: {physical_cards['fecha_emision'].min()} → {physical_cards['fecha_emision'].max()}")
print(f"Activation date range: {physical_cards['fecha_activacion'].min()} → {physical_cards['fecha_activacion'].max()}")

Physical cards: 5,382
Customers with physical card: 5,382
Issue date range: 2026-05-01 00:00:00 → 2026-07-01 00:00:00
Activation date range: 2026-05-02 00:00:00 → 2026-07-01 00:00:00


In [19]:
# Solapamiento campaña vs. tarjeta física
# Compara los clientes contactados por la campaña con quienes efectivamente recibieron tarjeta física.

campaign_customers = set(pilot_campaign["cliente_id"].unique())
physical_customers = set(physical_cards["cliente_id"].unique())

print(f"Campaign customers: {len(campaign_customers):,}")
print(f"Physical card customers: {len(physical_customers):,}")
print(f"Campaign + physical: {len(campaign_customers & physical_customers):,}")
print(f"Campaign without physical: {len(campaign_customers - physical_customers):,}")
print(f"Physical without campaign: {len(physical_customers - campaign_customers):,}")

Campaign customers: 5,402
Physical card customers: 5,382
Campaign + physical: 2,352
Campaign without physical: 3,050
Physical without campaign: 3,030


In [37]:
# Distribución temporal de la campaña
# Permite identificar si los contactos asociados al piloto se concentran en 2026.

campaign_monthly = (
    pilot_campaign
    .assign(month=pilot_campaign["fecha_contacto"].dt.to_period("M"))
    .groupby("month")
    .size()
    .reset_index(name="contacts")
)

display(campaign_monthly.tail(10))

,month,contacts
56,2025-09,107
57,2025-10,97
58,2025-11,99
59,2025-12,123
60,2026-01,121
61,2026-02,105
62,2026-03,103
63,2026-04,107
64,2026-05,129
65,2026-06,108


In [47]:
# Alineación entre campaña y emisión física
# Evalúa si los clientes con tarjeta física tuvieron contactos de campaña cercanos a su emisión.

physical_with_issue = (
    physical_cards
    .dropna(subset=["fecha_emision"])
    [["cliente_id", "fecha_emision"]]
    .drop_duplicates("cliente_id")
)

campaign_dates = (
    pilot_campaign
    .dropna(subset=["fecha_contacto"])
    [["cliente_id", "fecha_contacto"]]
)

campaign_issue_match = (
    physical_with_issue
    .merge(
        campaign_dates,
        on="cliente_id",
        how="inner",
    )
)

campaign_issue_match["days_before_issue"] = (
    campaign_issue_match["fecha_emision"]
    - campaign_issue_match["fecha_contacto"]
).dt.days

# Contacto previo más cercano a la emisión por cliente
closest_prior_contact = (
    campaign_issue_match[
        campaign_issue_match["days_before_issue"] >= 0
    ]
    .groupby("cliente_id", as_index=False)["days_before_issue"]
    .min()
)

campaign_alignment = (
    physical_with_issue
    .merge(
        closest_prior_contact,
        on="cliente_id",
        how="left",
    )
)

# Métricas de alineación
alignment_summary = pd.DataFrame({
    "metric": [
        "physical_customers_with_issue_date",
        "campaign_overlap",
        "prior_campaign_contact",
        "contact_within_30_days",
        "contact_within_60_days",
        "contact_within_90_days",
    ],
    "customers": [
        len(campaign_alignment),
        campaign_alignment["cliente_id"].isin(
            campaign_issue_match["cliente_id"]
        ).sum(),
        campaign_alignment["days_before_issue"].notna().sum(),
        campaign_alignment["days_before_issue"].between(0, 30).sum(),
        campaign_alignment["days_before_issue"].between(0, 60).sum(),
        campaign_alignment["days_before_issue"].between(0, 90).sum(),
    ],
})

alignment_summary["pct_physical"] = (
    alignment_summary["customers"]
    / len(campaign_alignment)
)

display(alignment_summary)

,metric,customers,pct_physical
0,physical_customers_with_issue_date,5381,1.000000
1,campaign_overlap,2352,0.437093
2,prior_campaign_contact,2297,0.426872
3,contact_within_30_days,55,0.010221
4,contact_within_60_days,98,0.018212
5,contact_within_90_days,146,0.027133


In [48]:
# Distancia del contacto previo más cercano
# Resume cuántos días antes de la emisión ocurrió la interacción más próxima.

display(
    campaign_alignment["days_before_issue"]
    .dropna()
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95])
    .to_frame()
)

,days_before_issue
count,2297.000000
mean,870.270353
std,555.848989
min,0.000000
25%,395.000000
50%,812.000000
75%,1330.000000
90%,1694.200000
95%,1817.200000
max,1984.000000


In [33]:
# Cobertura temporal de transacciones
# Determina el horizonte disponible para construir ventanas pre y post tratamiento.

print(f"Transaction start: {transactions['fecha'].min()}")
print(f"Transaction end: {transactions['fecha'].max()}")

Transaction start: 2019-12-01 00:00:00
Transaction end: 2026-09-28 00:00:00


In [34]:
# Distribución temporal de emisiones físicas
# Revisa cómo se distribuyó la entrega de tarjetas durante el piloto.

physical_issue_monthly = (
    physical_cards
    .assign(month=physical_cards["fecha_emision"].dt.to_period("M"))
    .groupby("month")
    .size()
    .reset_index(name="physical_cards")
)

display(physical_issue_monthly)

,month,physical_cards
0,2026-05,5002
1,2026-06,320
2,2026-07,59


In [35]:
# Distribución temporal de activaciones físicas
# Revisa cuándo se activaron las tarjetas físicas que sí registran activación.

physical_activation_monthly = (
    physical_cards
    .dropna(subset=["fecha_activacion"])
    .assign(month=lambda df: df["fecha_activacion"].dt.to_period("M"))
    .groupby("month")
    .size()
    .reset_index(name="activated_physical_cards")
)

display(physical_activation_monthly)

,month,activated_physical_cards
0,2026-05,2346
1,2026-06,108
2,2026-07,119


In [36]:
# Tiempo entre emisión y activación
# Valida la secuencia temporal y cuantifica cuánto tarda una tarjeta física en activarse.

physical_activated = (
    physical_cards
    .dropna(subset=["fecha_activacion"])
    .copy()
)

physical_activated["activation_delay_days"] = (
    physical_activated["fecha_activacion"]
    - physical_activated["fecha_emision"]
).dt.days

print(
    f"Activation before issue: "
    f"{(physical_activated['activation_delay_days'] < 0).sum():,}"
)

display(
    physical_activated["activation_delay_days"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95])
    .to_frame()
)

Activation before issue: 0


,activation_delay_days
count,2573.000000
mean,12.116984
std,7.083752
min,0.000000
25%,6.000000
50%,12.000000
75%,18.000000
90%,22.000000
95%,23.000000
max,24.000000


In [38]:
# Cobertura disponible posterior a la emisión
# Verifica que todos los clientes tratados cuenten con una ventana post comparable.

last_transaction_date = transactions["fecha"].max()

physical_cards["available_post_days"] = (
    last_transaction_date - physical_cards["fecha_emision"]
).dt.days

display(physical_cards["available_post_days"].describe())

print(
    "Physical cards with less than 60 post days:",
    (physical_cards["available_post_days"] < 60).sum(),
)

count    5381.000000
mean      145.830701
std        13.620103
min        89.000000
25%       150.000000
50%       150.000000
75%       150.000000
max       150.000000
Name: available_post_days, dtype: float64

Physical cards with less than 60 post days: 0


In [39]:
# Fecha índice del grupo control
# Utiliza la mediana de emisión del grupo tratado como referencia temporal común.

treated_customer_ids = card_ownership.loc[
    card_ownership["card_group"].eq("physical_and_virtual"),
    "cliente_id",
]

treated_physical_cards = physical_cards[
    physical_cards["cliente_id"].isin(treated_customer_ids)
].copy()

control_index_date = treated_physical_cards["fecha_emision"].median()

print(f"Control index date: {control_index_date}")

Control index date: 2026-05-01 00:00:00


#### Observaciones

- La campaña `piloto tarjeta fisica q2` no presenta una correspondencia suficiente con la emisión efectiva de tarjetas físicas. De los 5,381 clientes con tarjeta física y fecha de emisión válida, únicamente 2,352 (43.7%) aparecen alguna vez entre los clientes de dicha campaña.

- La desalineación también es temporal. Solo 55 clientes con tarjeta física (1.02%) tuvieron un contacto de la campaña dentro de los 30 días previos a su emisión, 98 (1.82%) dentro de 60 días y 146 (2.71%) dentro de 90 días. Entre los clientes con algún contacto previo, la mediana de separación respecto a la emisión fue de 812 días.

- En consecuencia, `marketing_interactions` no se utilizará para identificar la cohorte ni la fecha de tratamiento del piloto. La pertenencia al piloto se definirá a partir de la emisión efectiva registrada en `Cards`.

- Las tarjetas físicas fueron emitidas entre mayo y julio de 2026, con una fuerte concentración en mayo (5,002 de 5,382 tarjetas).

- Las transacciones presentan cobertura desde diciembre de 2019 hasta septiembre de 2026, proporcionando suficiente historia previa y posterior para analizar el comportamiento alrededor del piloto.

- Las 2,573 tarjetas físicas con fecha de activación presentan una secuencia temporal consistente: no existen activaciones anteriores a la emisión. El tiempo entre emisión y activación tiene una mediana de 12 días y un máximo de 24 días.

- La comparación principal se realizará entre clientes `physical_and_virtual` y `virtual_only`, ya que ambos grupos comparten la tenencia de tarjeta virtual y difieren principalmente por la incorporación de una tarjeta física. Los grupos `physical_only` y `no_card` se mantendrán como grupos secundarios.

- Para los clientes tratados, `t=0` corresponderá a la `fecha_emision` de la tarjeta física. Para el grupo `virtual_only`, se utilizará como fecha índice el 1 de mayo de 2026, correspondiente a la mediana de emisión del grupo tratado.

- Se utilizará una ventana de 60 días antes y 60 días después de la fecha índice. Todos los clientes con fecha de emisión física válida cuentan con al menos 60 días de cobertura posterior; una tarjeta física no dispone de `fecha_emision` válida y se excluirá únicamente de los análisis temporales.

- La comparación es observacional: utilizar `physical_and_virtual` frente a `virtual_only` reduce diferencias asociadas a la estructura de producto, pero no garantiza comparabilidad completa entre ambos grupos. La existencia de posibles diferencias previas se evaluará explícitamente antes de interpretar el desempeño post-piloto.

### 1.3 Comparabilidad pre-piloto

In [40]:
# Cohortes para evaluación pre-piloto
# Define tratados y controles utilizando únicamente información previa al piloto.

valid_customer_ids = set(customers["cliente_id"])

treated_ids = set(
    card_ownership.loc[
        card_ownership["card_group"].eq("physical_and_virtual"),
        "cliente_id",
    ]
) & valid_customer_ids

control_ids = set(
    card_ownership.loc[
        card_ownership["card_group"].eq("virtual_only"),
        "cliente_id",
    ]
) & valid_customer_ids

treated_index = (
    physical_cards[
        physical_cards["cliente_id"].isin(treated_ids)
    ]
    .groupby("cliente_id", as_index=False)["fecha_emision"]
    .min()
    .rename(columns={"fecha_emision": "index_date"})
)

treated_index["group"] = "treated"

control_index = pd.DataFrame({
    "cliente_id": sorted(control_ids),
    "index_date": pd.Timestamp("2026-05-01"),
    "group": "control",
})

cohort = pd.concat(
    [treated_index, control_index],
    ignore_index=True,
)

display(cohort["group"].value_counts())

group
control    6148
treated    4894
Name: count, dtype: int64

In [58]:
# Comportamiento transaccional pre-piloto
# Calcula actividad durante los 60 días anteriores a la fecha índice.

transactions_pre = (
    transactions[
        transactions["cliente_id"].isin(cohort["cliente_id"])
    ]
    .merge(
        cohort,
        on="cliente_id",
        how="inner",
    )
)

transactions_pre["days_from_index"] = (
    transactions_pre["fecha"]
    - transactions_pre["index_date"]
).dt.days

transactions_pre = transactions_pre[
    transactions_pre["days_from_index"].between(-60, -1)
].copy()

transactions_pre["monto_num"] = pd.to_numeric(
    transactions_pre["monto"],
    errors="coerce",
)

# Volumen transaccional pre-piloto
# Usa la magnitud del monto para mantener el mismo criterio de la sección 2.2.

transactions_pre["amount_volume"] = (
    transactions_pre["monto_num"].abs()
)

pre_metrics = (
    transactions_pre
    .groupby("cliente_id", as_index=False)
    .agg(
        pre_tx_count=("transaccion_id", "count"),
        pre_total_amount=("amount_volume", "sum"),
        pre_active_days=("fecha", "nunique"),
    )
)

pre_metrics = (
    cohort
    .merge(
        pre_metrics,
        on="cliente_id",
        how="left",
    )
)

pre_metrics[
    ["pre_tx_count", "pre_total_amount", "pre_active_days"]
] = pre_metrics[
    ["pre_tx_count", "pre_total_amount", "pre_active_days"]
].fillna(0)

In [59]:
# Comparabilidad de grupos
# Contrasta media y mediana de actividad previa entre tratados y controles.

pre_comparison = (
    pre_metrics
    .groupby("group")
    .agg(
        customers=("cliente_id", "nunique"),
        mean_tx=("pre_tx_count", "mean"),
        median_tx=("pre_tx_count", "median"),
        mean_amount=("pre_total_amount", "mean"),
        median_amount=("pre_total_amount", "median"),
        mean_active_days=("pre_active_days", "mean"),
    )
    .round(2)
)

display(pre_comparison)

,customers,mean_tx,median_tx,mean_amount,median_amount,mean_active_days
group,,,,,,
control,6148,1.34,1.0,40.08,13.42,1.30
treated,4894,1.82,1.0,54.56,16.73,1.64


In [60]:
# Diferencia estandarizada
# Cuantifica el desbalance previo entre tratados y controles.

def standardized_mean_difference(data, column):
    treated = data.loc[data["group"].eq("treated"), column]
    control = data.loc[data["group"].eq("control"), column]

    pooled_std = (
        (treated.var() + control.var()) / 2
    ) ** 0.5

    return (
        (treated.mean() - control.mean()) / pooled_std
        if pooled_std != 0
        else 0
    )


balance_check = pd.Series({
    column: standardized_mean_difference(pre_metrics, column)
    for column in [
        "pre_tx_count",
        "pre_total_amount",
        "pre_active_days",
    ]
}, name="smd").to_frame()

display(balance_check.round(3))

,smd
pre_tx_count,0.177
pre_total_amount,0.169
pre_active_days,0.151


#### Observaciones

- Antes del piloto, el grupo tratado ya presentaba una actividad transaccional superior al grupo de comparación. En los 60 días previos registró en promedio 1.82 transacciones frente a 1.34 en el grupo control, así como 1.64 días activos frente a 1.30.

- El volumen transaccionado previo también fue superior en el grupo tratado, con un promedio de 54.56 frente a 40.08 en el grupo control. Para esta métrica se utiliza la magnitud absoluta del monto, manteniendo el mismo criterio empleado posteriormente en el análisis de desempeño.

- Las diferencias estandarizadas se encuentran entre 0.151 y 0.177, evidenciando un desbalance previo moderado entre ambos grupos.

- Estos resultados no permiten determinar el mecanismo mediante el cual fueron seleccionados los clientes del piloto, pero sí muestran que los grupos no eran completamente comparables antes de la emisión de las tarjetas físicas.

- En consecuencia, el desempeño posterior no se interpretará únicamente a partir de diferencias absolutas entre grupos. Se priorizará la comparación del cambio pre/post de cada cohorte, utilizando el grupo `virtual_only` como referencia para contextualizar la evolución observada en los clientes tratados.

## 2. Desempenio del Piloto

### 2.1 Adopcion y Activacion

In [44]:
# Adopción y activación
# Resume la emisión y activación observada de tarjetas físicas.

physical_performance = physical_cards.copy()

physical_performance["is_activated"] = (
    physical_performance["fecha_activacion"].notna()
)

physical_summary = pd.DataFrame({
    "metric": [
        "physical_cards_issued",
        "physical_cards_activated",
        "activation_rate",
    ],
    "value": [
        len(physical_performance),
        physical_performance["is_activated"].sum(),
        physical_performance["is_activated"].mean(),
    ],
})

display(physical_summary)

,metric,value
0,physical_cards_issued,5382.000000
1,physical_cards_activated,2573.000000
2,activation_rate,0.478075


In [45]:
# Activación por cohorte de emisión
# Compara la tasa de activación según el mes en que se emitió la tarjeta física.

physical_performance["issue_month"] = (
    physical_performance["fecha_emision"].dt.to_period("M")
)

activation_by_issue_month = (
    physical_performance
    .dropna(subset=["issue_month"])
    .groupby("issue_month")
    .agg(
        issued=("cliente_id", "size"),
        activated=("is_activated", "sum"),
    )
    .reset_index()
)

activation_by_issue_month["activation_rate"] = (
    activation_by_issue_month["activated"]
    / activation_by_issue_month["issued"]
)

display(activation_by_issue_month)

,issue_month,issued,activated,activation_rate
0,2026-05,5002,2386,0.477009
1,2026-06,320,163,0.509375
2,2026-07,59,24,0.406780


In [46]:
# Tiempo hasta activación
# Resume cuánto tarda una tarjeta física en registrar activación.

activation_delay_summary = (
    physical_activated["activation_delay_days"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95])
    .to_frame()
)

display(activation_delay_summary)

,activation_delay_days
count,2573.000000
mean,12.116984
std,7.083752
min,0.000000
25%,6.000000
50%,12.000000
75%,18.000000
90%,22.000000
95%,23.000000
max,24.000000


#### Observaciones

- Se emitieron 5,382 tarjetas físicas durante el piloto, de las cuales 2,573 registran una fecha de activación. Esto representa una tasa de activación observada de 47.81%.

- La tasa de activación se mantiene relativamente estable entre las principales cohortes de emisión: 47.70% para mayo y 50.94% para junio. La cohorte de julio presenta una tasa menor (40.68%), aunque corresponde únicamente a 59 tarjetas, por lo que su resultado debe interpretarse con cautela debido al reducido tamaño de muestra.

- Entre las tarjetas que sí registraron activación, el tiempo medio entre emisión y activación fue de 12.12 días y la mediana de 12 días.

- El 75% de las activaciones ocurrió dentro de los primeros 18 días posteriores a la emisión y el 95% dentro de los primeros 23 días.

- La activación constituye uno de los principales puntos de pérdida observados en el proceso: más de la mitad de las tarjetas físicas emitidas no registra activación. Sin embargo, al no disponer de una meta o benchmark de activación definido por negocio, este resultado se reporta como desempeño observado y no se clasifica por sí solo como éxito o fracaso del piloto.

### 2.2 Comportamiento Transaccional

In [49]:
# Semántica del monto por tipo de transacción
# Revisa signos y devoluciones antes de construir las métricas de volumen.

transactions["monto_num"] = pd.to_numeric(
    transactions["monto"],
    errors="coerce",
)

amount_check = (
    transactions
    .groupby("tipo_transaccion")
    .agg(
        transactions=("transaccion_id", "count"),
        min_amount=("monto_num", "min"),
        max_amount=("monto_num", "max"),
        negative_amounts=("monto_num", lambda x: (x < 0).sum()),
        refunds=("es_devolucion", lambda x: x.eq(True).sum()),
    )
    .reset_index()
)

amount_check["negative_pct"] = (
    amount_check["negative_amounts"]
    / amount_check["transactions"]
)

amount_check["refund_pct"] = (
    amount_check["refunds"]
    / amount_check["transactions"]
)

display(amount_check)

,tipo_transaccion,transactions,min_amount,max_amount,negative_amounts,refunds,negative_pct,refund_pct
0,cash_in,52051,-215.01,55693.44,525,12301,0.010086,0.236326
1,cash_out,34917,-173.55,56648.74,360,8362,0.010310,0.239482
2,compra_tarjeta,19017,-51.42,55991.25,172,4505,0.009045,0.236893
3,p2p_in,26108,-108.61,50911.46,258,6242,0.009882,0.239084
4,p2p_out,26073,-98.96,49155.47,259,6294,0.009934,0.241399
5,pago_servicio,8770,-133.39,167.05,95,2053,0.010832,0.234094
6,recarga_celular,17364,-125.02,208.95,181,4074,0.010424,0.234623
7,remesa,8715,-125.44,47023.21,80,2042,0.009180,0.234309


In [50]:
# Relación entre montos negativos y devoluciones
# Verifica si ambas señales representan el mismo fenómeno.

transactions["is_negative"] = transactions["monto_num"] < 0

display(
    pd.crosstab(
        transactions["is_negative"],
        transactions["es_devolucion"],
        margins=True,
    )
)

es_devolucion,False,True,All
is_negative,,,
False,145685,45400,191085
True,1457,473,1930
All,147142,45873,193015


In [51]:
# Distribución de montos
# Evalúa el peso de valores extremos antes de agregar volumen transaccional.

display(
    transactions["monto_num"]
    .describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99, 0.995])
    .to_frame()
)

,monto_num
count,193015.000000
mean,31.533782
std,404.884137
min,-215.010000
50%,23.520000
75%,38.540000
90%,56.520000
95%,69.280000
99%,97.030000
99.5%,109.340000


In [52]:
# Métricas transaccionales PRE y POST
# Compara 60 días antes y después de la fecha índice de cada cliente.

analysis_transactions = (
    transactions[
        transactions["cliente_id"].isin(cohort["cliente_id"])
    ]
    .merge(
        cohort,
        on="cliente_id",
        how="inner",
    )
    .copy()
)

analysis_transactions["days_from_index"] = (
    analysis_transactions["fecha"]
    - analysis_transactions["index_date"]
).dt.days

analysis_transactions["amount_volume"] = (
    analysis_transactions["monto_num"].abs()
)

analysis_transactions["period"] = pd.NA

analysis_transactions.loc[
    analysis_transactions["days_from_index"].between(-60, -1),
    "period",
] = "pre"

analysis_transactions.loc[
    analysis_transactions["days_from_index"].between(0, 59),
    "period",
] = "post"

analysis_transactions = analysis_transactions[
    analysis_transactions["period"].notna()
].copy()

In [53]:
# Comportamiento por cliente y periodo
# Resume frecuencia, volumen y días con actividad.

customer_period_metrics = (
    analysis_transactions
    .groupby(
        ["cliente_id", "group", "period"],
        as_index=False,
    )
    .agg(
        tx_count=("transaccion_id", "count"),
        total_amount=("amount_volume", "sum"),
        active_days=("fecha", "nunique"),
    )
)

In [54]:
# Incluye clientes sin actividad
# Evita excluir silenciosamente clientes con cero transacciones.

periods = pd.DataFrame({"period": ["pre", "post"]})

customer_grid = (
    cohort[["cliente_id", "group"]]
    .merge(periods, how="cross")
)

customer_period_metrics = (
    customer_grid
    .merge(
        customer_period_metrics,
        on=["cliente_id", "group", "period"],
        how="left",
    )
)

customer_period_metrics[
    ["tx_count", "total_amount", "active_days"]
] = customer_period_metrics[
    ["tx_count", "total_amount", "active_days"]
].fillna(0)

In [55]:
# Resumen PRE vs POST
# Compara el comportamiento observado de ambas cohortes.

transaction_summary = (
    customer_period_metrics
    .groupby(["group", "period"])
    .agg(
        customers=("cliente_id", "nunique"),
        mean_tx=("tx_count", "mean"),
        median_tx=("tx_count", "median"),
        mean_amount=("total_amount", "mean"),
        median_amount=("total_amount", "median"),
        mean_active_days=("active_days", "mean"),
        median_active_days=("active_days", "median"),
    )
    .round(2)
)

display(transaction_summary)

customers  mean_tx  median_tx  mean_amount  median_amount  \
group   period                                                              
control post         6148     1.94        1.0        57.91          18.66   
        pre          6148     1.34        1.0        40.08          13.42   
treated post         4894     4.60        3.0       103.09          65.77   
        pre          4894     1.82        1.0        54.56          16.73   

                mean_active_days  median_active_days  
group   period                                        
control post                1.75                 1.0  
        pre                 1.30                 1.0  
treated post                4.12                 3.0  
        pre                 1.64                 1.0

In [56]:
# Cambio PRE vs POST por cohorte
# Cuantifica la evolución de cada grupo y su diferencia relativa.

summary_wide = (
    transaction_summary
    .reset_index()
    .pivot(
        index="group",
        columns="period",
        values=[
            "mean_tx",
            "mean_amount",
            "mean_active_days",
        ],
    )
)

delta_summary = pd.DataFrame({
    "tx_delta": (
        summary_wide["mean_tx"]["post"]
        - summary_wide["mean_tx"]["pre"]
    ),
    "amount_delta": (
        summary_wide["mean_amount"]["post"]
        - summary_wide["mean_amount"]["pre"]
    ),
    "active_days_delta": (
        summary_wide["mean_active_days"]["post"]
        - summary_wide["mean_active_days"]["pre"]
    ),
})

display(delta_summary.round(2))

,tx_delta,amount_delta,active_days_delta
group,,,
control,0.60,17.83,0.45
treated,2.78,48.53,2.48


#### Observaciones

- Ambos grupos incrementaron su actividad transaccional en los 60 días posteriores a la fecha índice, lo que indica que parte de la evolución observada puede responder a factores temporales comunes y no exclusivamente a la emisión de la tarjeta física.

- El incremento fue considerablemente mayor en el grupo tratado. El promedio de transacciones por cliente pasó de 1.82 a 4.60, equivalente a un aumento absoluto de 2.78 transacciones, mientras que el grupo control pasó de 1.34 a 1.94, con un incremento de 0.60.

- El volumen transaccionado promedio también presentó una evolución superior en los tratados: aumentó de 54.56 a 103.09 por cliente, con un incremento de 48.53, frente a un aumento de 17.83 en el grupo control. Esta métrica representa volumen transaccional utilizando la magnitud absoluta del monto y no flujo monetario neto.

- Los días con actividad aumentaron de 1.64 a 4.12 en el grupo tratado, un incremento de 2.48 días activos, frente a un aumento de 0.45 días en el grupo control.

- Las medianas muestran el mismo patrón y reducen la preocupación de que el resultado esté explicado únicamente por valores extremos. En el grupo tratado, la mediana de transacciones y días activos pasó de 1 a 3, mientras que permaneció en 1 para el grupo control. La mediana del volumen transaccionado también aumentó de 16.73 a 65.77 entre los tratados, frente a 13.42 a 18.66 en el control.

- En conjunto, los clientes que recibieron una tarjeta física presentan una mejora transaccional posterior sustancialmente mayor que la observada en el grupo `virtual_only`. Sin embargo, debido al desbalance existente antes del piloto y a que el mecanismo de asignación no está documentado, esta diferencia se interpreta como una asociación con la emisión de la tarjeta física y no como una estimación causal de su efecto.

## 3. Conclusiones del Piloto

### Resumen ejecutivo

> **Resultado general:** El piloto muestra una señal favorable de mayor actividad transaccional posterior entre los clientes con tarjeta física, aunque la comparación no permite atribuir causalidad.

| Indicador | Resultado | Lectura |
|---|---:|---|
| **Activación física** | **47.81%** | 2,573 de 5,382 tarjetas emitidas registran activación |
| **Δ transacciones / cliente** | **+2.78** vs. +0.60 control | Mayor crecimiento de frecuencia en tratados |
| **Δ días activos / cliente** | **+2.48** vs. +0.45 control | Mayor recurrencia transaccional |
| **Δ volumen / cliente** | **+48.53** vs. +17.83 control | Mayor crecimiento del volumen transaccionado |

**Lectura de negocio:**  
La principal oportunidad se encuentra en la **activación**, ya que más de la mitad de las tarjetas físicas emitidas no registra este evento. Entre los clientes tratados, el comportamiento posterior muestra un crecimiento transaccional considerablemente superior al grupo `virtual_only`.

> **Cautela metodológica:** Los grupos presentaban diferencias antes del piloto (SMD 0.15–0.18) y no se dispone del mecanismo de asignación de las tarjetas físicas. Por ello, los resultados representan una **asociación favorable**, no un efecto causal estimado.

### Conclusiones

- La activación representa el principal punto de pérdida observado. Entre las tarjetas que sí registraron activación, la mediana fue de 12 días desde la emisión.

- El grupo tratado ya presentaba mayor actividad antes del piloto, por lo que el análisis priorizó el cambio pre/post de cada cohorte en lugar de comparar únicamente sus niveles posteriores.

- Los clientes con tarjeta física mostraron una mejora transaccional posterior sustancialmente mayor que el grupo `virtual_only`, tanto en frecuencia como en días activos y volumen transaccionado.

- Para una siguiente ola, las principales oportunidades son mejorar la conversión entre emisión y activación y priorizar clientes con mayor probabilidad de activar y utilizar efectivamente la tarjeta física.